# Setup: Groups and Users for HIPAA Demo

**Purpose:** This notebook programmatically creates all the security groups and (optionally) test users needed for the HIPAA-Compliant Patient Analytics demo.

> **Estimated runtime:** < 1 minute

In [0]:
# =============================================================================
# Step 1: Initialize the Databricks SDK WorkspaceClient
# On serverless compute, authentication is handled automatically.
# =============================================================================

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.iam import Group, ComplexValue, Patch, PatchSchema, PatchOp

w = WorkspaceClient()

# Verify connectivity by getting the current user
current_user = w.current_user.me()
print(f"✅ Connected as: {current_user.user_name}")
print(f"   Display name: {current_user.display_name}")
print(f"   User ID: {current_user.id}")

In [0]:
# =============================================================================
# Step 2: Define all groups needed for the HIPAA demo
# These map directly to the IS_MEMBER() checks in masking/filter functions
# =============================================================================

# Access tier groups (used by column masking functions)
access_tier_groups = {
    "phi_authorized":       "Full PHI access - clinical staff, treating physicians",
    "research_analysts":    "Partial masking - IRB-approved researchers",
    "billing_team":         "Name + insurance access - revenue cycle staff",
    "compliance_officers":  "Full read access - audit and compliance team",
}

# Regional groups (used by row-level security on encounters)
regional_groups = {
    "northeast_team": "Access to Northeast region data only",
    "southeast_team": "Access to Southeast region data only",
    "midwest_team":   "Access to Midwest region data only",
    "southwest_team": "Access to Southwest region data only",
    "west_team":      "Access to West region data only",
}

# Department groups (used by department row filter)
department_groups = {
    "dept_cardiology":    "Access to Cardiology department data",
    "dept_oncology":      "Access to Oncology department data",
    "dept_neurology":     "Access to Neurology department data",
    "dept_emergency":     "Access to Emergency department data",
    "dept_surgery":       "Access to Surgery department data",
    "dept_pediatrics":    "Access to Pediatrics department data",
    "dept_orthopedics":   "Access to Orthopedics department data",
    "dept_internal_med":  "Access to Internal Medicine department data",
}

all_groups = {**access_tier_groups, **regional_groups, **department_groups}
print(f"📋 Total groups to create: {len(all_groups)}")
print(f"   Access tier: {len(access_tier_groups)}")
print(f"   Regional:    {len(regional_groups)}")
print(f"   Department:  {len(department_groups)}")

In [0]:
# =============================================================================
# Step 3: Create all groups using the Databricks SDK
# Safely skips groups that already exist
# =============================================================================

# Get existing groups to avoid duplicates
existing_groups = {g.display_name: g for g in w.groups.list()}
print(f"🔍 Found {len(existing_groups)} existing groups in workspace\n")

created_count = 0
skipped_count = 0
error_count = 0

for group_name, description in all_groups.items():
    if group_name in existing_groups:
        print(f"  ⏭️  SKIP  '{group_name}' (already exists, id: {existing_groups[group_name].id})")
        skipped_count += 1
    else:
        try:
            new_group = w.groups.create(
                display_name=group_name,
            )
            print(f"  ✅ CREATED '{group_name}' (id: {new_group.id})")
            existing_groups[group_name] = new_group
            created_count += 1
        except Exception as e:
            print(f"  ❌ ERROR  '{group_name}': {e}")
            error_count += 1

print(f"\n{'='*60}")
print(f"🎉 Summary: {created_count} created, {skipped_count} skipped, {error_count} errors")
print(f"{'='*60}")

In [0]:
# =============================================================================
# Step 4: Verify all required groups exist
# =============================================================================

# Refresh the group list
current_groups = {g.display_name: g for g in w.groups.list()}

print("\n\U0001f50d Group Verification Report")
print("=" * 70)
print(f"{'Group Name':<25} {'Status':<12} {'Group ID':<15} {'Description'}")
print("-" * 70)

all_ok = True
for group_name, description in all_groups.items():
    if group_name in current_groups:
        gid = current_groups[group_name].id
        print(f"{group_name:<25} \u2705 EXISTS   {gid:<15} {description}")
    else:
        print(f"{group_name:<25} \u274c MISSING  {'N/A':<15} {description}")
        all_ok = False

print("-" * 70)
if all_ok:
    print("\u2705 All required groups are present!")
else:
    print("\u26a0\ufe0f Some groups are missing - re-run Step 3")

In [0]:
# =============================================================================
# Step 5: Remove the current user from ALL demo groups
# Run this to reset to the "default" (most restrictive) masking level
# =============================================================================

current_user = w.current_user.me()
user_id = current_user.id
user_email = current_user.user_name
current_groups = {g.display_name: g for g in w.groups.list()}

print(f"Removing '{user_email}' from all demo groups...\n")

removed_count = 0
for group_name in all_groups:
    if group_name in current_groups:
        try:
            group_detail = w.groups.get(current_groups[group_name].id)
            member_ids = [m.value for m in (group_detail.members or [])]
            
            if user_id in member_ids:
                w.groups.patch(
                    id=current_groups[group_name].id,
                    schemas=[PatchSchema.URN_IETF_PARAMS_SCIM_API_MESSAGES_2_0_PATCH_OP],
                    operations=[
                        Patch(
                            op=PatchOp.REMOVE,
                            path=f'members[value eq "{user_id}"]'
                        )
                    ]
                )
                print(f"  \u2705 Removed from '{group_name}'")
                removed_count += 1
            else:
                print(f"  \u23ed\ufe0f  Not in '{group_name}' (skipped)")
        except Exception as e:
            print(f"  \u274c Error with '{group_name}': {e}")

print(f"\n{'='*50}")
print(f"Removed from {removed_count} group(s).")
print(f"You now have the DEFAULT (most restrictive) masking level.")
print(f"{'='*50}")